In [57]:
import json

jsonl_file = "/qumulo/shared_data/aofei_summer/RegTok/data/RegAlign_GPT5_mini.jsonl"
with open(jsonl_file, "r") as f:
    data = [json.loads(line) for line in f]

jsonl_file_02 = "/qumulo/shared_data/aofei_summer/RegTok/data/RegAlign_GPT5_mini_02.jsonl"
with open(jsonl_file_02, "r") as f:
    data_02 = [json.loads(line) for line in f]

data = data + data_02

In [ ]:
# preprocess the captioning data
caption_path = 

In [58]:
len(data)

20000

In [73]:
caption_prompt_candidates = [
    "Generate a fine-grained detailed caption of this medical image. List main detected regions, and for each region include: bounding box coordinates, anatomical description, and the corresponding segmentation code.",
    "Provide a structured description of this image. Break it into regions and describe each with: (1) bounding box, (2) organ or structure, (3) segmentation code, (4) a short caption of visual appearance.",
    "For this image, output a comprehensive region-level caption. Each region should be specified with its bounding box, its relative location (e.g., upper left, central, lower right), the anatomical or pathological structure, and the quantizer/segmentation code.",
    "Create a report-style caption for the image. Divide the image into regions and for each region provide: bounding box coordinates, description of the tissue or organ, and the assigned segmentation code.",
    "Generate a list of all relevant regions in this image. For each region, provide: {bounding box, description, segmentation code}. Cover all major visible structures.",
    "Describe this medical image at a fine-grained level. Identify every important region and output for each: bounding box, relative anatomical location, detailed caption, and its segmentation code."
]

In [66]:
# preprocess to MLLM training style
"""
target format:
{
    "id": id,
    "image": image_path,
    "conversations": [
    {
        "from":  "human", "value": user_message
    },
    {
        "from":  "assistant", "value": assistant_message
    }, ...
    ],
    "mask_files": {
        "mask_id": mask_file_path,
    },
    "mask_orders": [] # the ids of mentioned masks one by one, separated by commas
}
"""

def preprocess_data(item):
    """
    Return two items: caption_item and conv_item in the target MLLM format.
    This implementation handles the example input shape where:
    - item['image_file'] or item['image'] holds image path
    - item['caption'] may be a dict with 'text' and 'mask_ids_order'
    - item['dialogues'] is a list of dicts with 'User', 'Assistant', and 'mask_ids_order'
    - item['mask_code'] is a list of [id, label, path]
    """
    # id and image
    id_ = item['image_id']
    image = item.get('image_file') or item.get('image') or item.get('image_path') or item.get('img') or ''

    # Build mask_files from several possible formats
    mask_files = {}
    mask_codes = {}
    if 'mask_code' in item and isinstance(item['mask_code'], list):
        for entry in item['mask_code']:
            # expected entry: [id, label, path]
            try:
                mid = int(entry[0])
                code = entry[1] if len(entry) > 1 else None
                path = entry[2] if len(entry) > 2 else None
                if path:
                    mask_files[str(mid)] = path
                    mask_codes[str(mid)] = code
            except Exception:
                continue
    else:
        mf = item.get('mask_files') or item.get('masks') or {}
        if isinstance(mf, list):
            mask_files = {str(i): p for i, p in enumerate(mf)}
        elif isinstance(mf, dict):
            # convert keys to strings
            mask_files = {str(k): v for k, v in mf.items()}

    # Determine mask_orders: prefer caption-specific, then top-level, then collect from dialogues
    mask_orders = []
    if isinstance(item.get('caption'), dict) and 'mask_ids_order' in item['caption']:
        mask_orders = list(item['caption']['mask_ids_order'])
    elif 'mask_ids_order' in item:
        mask_orders = list(item['mask_ids_order']) if isinstance(item['mask_ids_order'], (list, tuple)) else [item['mask_ids_order']]

    # Caption-style item
    caption_text = ''
    if isinstance(item.get('caption'), dict):
        caption_text = item['caption'].get('text') or item['caption'].get('caption') or ''
    else:
        caption_text = item.get('caption') or item.get('summary') or item.get('description') or item.get('answer') or ''
    import random
    caption_question = random.choice(caption_prompt_candidates)
    caption_item = {
        'id': f"{id_}_caption" if id_ else 'caption',
        'image': image,
        'conversations': [
            {'from': 'human', 'value': caption_question},
            {'from': 'assistant', 'value': caption_text},
        ],
        'mask_files': mask_files,
        'mask_codes': mask_codes,
        'mask_orders': mask_orders,
    }

    # Full conversations: normalize dialogues to the target format
    convs = []
    mask_orders_conv = []
    # If 'dialogues' exists (example uses this), iterate and convert
    if 'dialogues' in item and isinstance(item['dialogues'], list):
        for turn in item['dialogues']:
            if not isinstance(turn, dict):
                continue
            user = turn.get('User') or turn.get('user') or turn.get('question') or turn.get('prompt')
            assistant = turn.get('Assistant') or turn.get('assistant') or turn.get('answer') or turn.get('response')
            if user is not None:
                convs.append({'from': 'human', 'value': user})
            if assistant is not None:
                convs.append({'from': 'assistant', 'value': assistant})
            # extend global mask_orders with any ids mentioned in this turn (preserve order, uniqueness)
            if 'mask_ids_order' in turn and isinstance(turn['mask_ids_order'], (list, tuple)):
                for mid in turn['mask_ids_order']:
                    mask_orders_conv.append(mid)
    else:
        # Fallback: try single-turn keys
        q = item.get('question') or item.get('user') or item.get('prompt')
        a = item.get('answer') or item.get('response') or item.get('assistant') or item.get('caption')
        if q is not None:
            convs.append({'from': 'human', 'value': q})
        if a is not None:
            convs.append({'from': 'assistant', 'value': a})

    conv_item = {
        'id': id_,
        'image': image,
        'conversations': convs,
        'mask_files': mask_files,
        'mask_codes': mask_codes,
        'mask_orders': mask_orders_conv,
    }

    return caption_item, conv_item

# Example usage with your provided input (uncomment to test):
# example = { ... }
# caption_item, conv_item = preprocess_data(example)
# print(caption_item)
# print(conv_item)

In [67]:
# ...existing code...
import re
def check_conversation(item):
    conversations = item.get("conversations", [])
    text = " ".join(conv.get("value", "") for conv in conversations)
    # pattern captures modality_index and code_index from tokens like [M2_15]
    pattern = r'\[M(\d+)_(\d+)\]'
    matches = re.findall(pattern, text)  # returns list of (modality_index, code_index) tuples
    flag = len(matches) == len(item.get("mask_orders", []))
    # keep order and uniqueness
    # seen = set()
    # codes = []
    # for mod, code in matches:
    #     key = f"{mod}_{code}"
    #     if key in seen:
    #         continue
    #     seen.add(key)
    #     codes.append({
    #         "modality_index": int(mod),
    #         "code_index": int(code),
    #         "token": f"[M{mod}_{code}]"
    #     })

    return flag

def check_caption(item):
    captions = item.get("conversations", [])
    text = " ".join(capt.get("value", "") for capt in captions)
    pattern = r'\[M(\d+)_(\d+)\]'
    matches = re.findall(pattern, text)
    flag = len(matches) == len(item.get("mask_orders", []))

    if not flag:
        mask_codes = item.get("mask_codes", {})
        mask_order_ids = item.get("mask_orders", [])
        # print(mask_codes, mask_order_ids)
        if len(item.get("mask_orders", [])) < len(matches):
            # print("Mask Order IDs:", mask_order_ids)
            matched_Codes = [f"M{i[0]}_{i[1]}" for i in matches]
            id_to_Codes = [mask_codes.get(str(i), "") for i in mask_order_ids]
            # print(matched_Codes, id_to_Codes)
            if matched_Codes[:len(id_to_Codes)] == id_to_Codes:
                flag = True
                # print("Matched Codes:", matched_Codes)
    
    return flag

In [68]:
example_Data = preprocess_data(data[108])
print(example_Data)
check_caption(example_Data[0])

({'id': '108_caption', 'image': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task01_BrainTumour/train/BRATS_074_54_MRI-FLAIR_brain.png', 'conversations': [{'from': 'human', 'value': 'Provide a structured description of this image. Break it into regions and describe each with: (1) bounding box, (2) organ or structure, (3) segmentation code, (4) a short caption of visual appearance.'}, {'from': 'assistant', 'value': 'This brain MRI FLAIR sequence shows a region of edema [M10_27] with bounding box [0.517, 0.283, 0.266, 0.399], localized to the right‑central portion of the slice in image coordinates; the annotation specifically indicates FLAIR‑hyperintense edema in that region.'}], 'mask_files': {'24581': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task01_BrainTumour/train_mask/BRATS_074_54_MRI-FLAIR_brain_edema.png'}, 'mask_codes': {'24581': 'M10_27'}, 'mask_orders': [24581]}, {'id': 108, 'image': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task

True

In [74]:
correct_items_conversation = []
correct_items_caption = []
failed_items_conversation = []
failed_items_caption = []
for i in data:
    processed_i = preprocess_data(i)
    if check_conversation(processed_i[1]):
        correct_items_conversation.append(processed_i)
    else:
        failed_items_conversation.append(processed_i)
    if check_caption(processed_i[0]):
        correct_items_caption.append(processed_i)
    else:
        failed_items_caption.append(processed_i)
    

In [75]:
correct_items_caption[89], len(correct_items_caption)

(({'id': '89_caption',
   'image': '/qumulo/shared_data/aofei_summer/data/BiomedParse/CAMUS/CAMUS/train/patient0205_4CH_half_sequence_3_ultrasound_heart.png',
   'conversations': [{'from': 'human',
     'value': 'Create a report-style caption for the image. Divide the image into regions and for each region provide: bounding box coordinates, description of the tissue or organ, and the assigned segmentation code.'},
    {'from': 'assistant',
     'value': 'This cardiac ultrasound demonstrates the left ventricle [M3_9] with bounding box [0.42, 0.282, 0.234, 0.373], located in the upper-central portion of the image, and the left atrium [M3_4] at bbox [0.412, 0.639, 0.222, 0.218], which lies inferior to and partially overlapping the ventricle on the image. Masks referenced in order: 28656, 28657.'}],
   'mask_files': {'28656': '/qumulo/shared_data/aofei_summer/data/BiomedParse/CAMUS/CAMUS/train_mask/patient0205_4CH_half_sequence_3_ultrasound_heart_left+heart+ventricle.png',
    '28657': '/q

In [78]:
items_to_save[88]

{'id': '88_caption',
 'image': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train/amos_0104_55_CT_abdomen.png',
 'conversations': [{'from': 'human',
   'value': 'Generate a list of all relevant regions in this image. For each region, provide: {bounding box, description, segmentation code}. Cover all major visible structures.'},
  {'from': 'assistant',
   'value': 'This abdominal CT contains multiple annotated structures: the right kidney [M0_19] at bbox [0.227, 0.355, 0.132, 0.144] located on the image-left central region; the left kidney [M0_26] at bbox [0.568, 0.35, 0.171, 0.151] located on the image-right central region; the gallbladder [M0_29] at bbox [0.223, 0.643, 0.069, 0.069] inferior to the liver on the image-left lower quadrant; the liver region [M0_16] at bbox [0.117, 0.422, 0.146, 0.245] occupying the upper-left portion of the abdomen; the stomach [M0_8] at bbox [0.309, 0.607, 0.323, 0.144] across the lower-central to left-central abdomen; the postcav

In [76]:
# save this conversation item
items_to_save = [i[0] for i in correct_items_caption]
save_path = "/qumulo/shared_data/aofei_summer/data/RegAlign/Seg_caption_20k.json"
# items_to_save[0]
with open(save_path, "w") as f:
    json.dump(items_to_save, f)

In [80]:
# save this conversation item
items_to_save = [i[1] for i in correct_items_conversation]
print(len(items_to_save))
save_path = "/qumulo/shared_data/aofei_summer/data/RegAlign/instruction_tuning_17k.json"
# items_to_save[0]
with open(save_path, "w") as f:
    json.dump(items_to_save, f)

16912


In [82]:
items_to_save[88]

{'id': 100,
 'image': '/qumulo/shared_data/aofei_summer/data/BiomedParse/Radiography/Radiography/Normal/train/Normal-1_X-Ray_chest.png',
 'conversations': [{'from': 'human',
   'value': 'Can you segment the lungs on this chest X‑ray?'},
  {'from': 'assistant',
   'value': 'Yes. The overall lung field is annotated as [M4_21] with bbox [0.129, 0.141, 0.632, 0.671].'},
  {'from': 'human', 'value': 'Where is the right lung region?'},
  {'from': 'assistant',
   'value': 'The right lung region is labeled [M4_14] with bbox [0.129, 0.141, 0.632, 0.304]; it lies in the upper portion of the overall lung field [M4_21].'},
  {'from': 'human', 'value': 'And the left lung?'},
  {'from': 'assistant',
   'value': 'The left lung region is labeled [M4_12] with bbox [0.129, 0.508, 0.624, 0.304]; it occupies the lower portion of the overall lung field [M4_21]. Remember that chest X‑ray laterality in the image is mirror‑like relative to the patient.'},
  {'from': 'human', 'value': 'Do the left and right re

In [43]:
32*18

576

In [13]:
failed_items[0]

({'id': '11_caption',
  'image': '/qumulo/shared_data/aofei_summer/data/BiomedParse/COVID-QU-Ex/COVID-QU-Ex/train/covid_3393_X-Ray_chest.png',
  'conversations': [{'from': 'human', 'value': 'Describe the image.'},
   {'from': 'assistant',
    'value': "This chest radiograph shows the lung field [M4_25] occupying a large portion of the image (bbox: [0.074, 0.047, 0.851, 0.905]). There is a broader region labeled as infection/COVID-19 findings [M4_4] which overlaps the pulmonary fields (bbox: [0.074, 0.117, 0.851, 0.835]), indicating diffuse air‑space or interstitial involvement. The left lung region [M4_12] is annotated with bbox [0.098, 0.625, 0.827, 0.327] and the right lung region [M4_14] with bbox [0.074, 0.047, 0.792, 0.476]; both sit within the overall lung mask. Note: chest radiograph images can be presented mirrored relative to patient anatomy (the patient's left may appear on the viewer's right); the annotations here use the provided labels, and the image-coordinate locations a

In [15]:
# test the items with 1-1 codes and mask mapping
data[0]
s = 0
for i in data:
    mask_code_mapping = i['mask_code']
    codes = [j[1] for j in mask_code_mapping]
    if len(codes) == len(set(codes)):
        s+=1
    

In [16]:
s

8172

In [ ]:
# first filter out those data with correct number of id orders, then use the 1-1 codes and mask mapping to reconstruct the id orders
# the other data will be abandoned